<a href="https://colab.research.google.com/github/PanditPranav/WildAlertModels_Circumstances/blob/main/notebooks/01_30092024_explore-data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd

from pathlib import Path

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import torch
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
     print("cuda is not available")

NVIDIA A100-SXM4-80GB


In [4]:
data_dir = Path("/content/drive/MyDrive/WildAlertCOA/data/processed")
ckpt = "bert-base-uncased"

raw_data_dir = Path("/content/drive/MyDrive/WildAlertCOA/data/raw")
interim_data_dir = Path("/content/drive/MyDrive/WildAlertCOA/data/interim")

In [5]:
WNDP_API = ""

In [6]:
%%time
r = requests.get(WNDP_API)

CPU times: user 35.6 ms, sys: 12.7 ms, total: 48.3 ms
Wall time: 52.3 s


In [7]:
r = r.json()

In [8]:
df = pd.DataFrame(r)
df.drop('case_id', axis=1, inplace=True)
df.head(20)

,text,terms
0,"Mallard. In road, easy to catch, hit by car. ...",[Vehicle collision]
1,Grey Fox. Orphan,[Orphan]
2,Malay Spotted Dove. Cat attack,[Cat interaction]
3,"Mew Gull. Found on ground, in road, easy to ca...",[Vehicle collision]
4,Common Merganser. Orphans,[Orphan]
5,Common Merganser. Orphans,[Orphan]
6,Red-billed gull. Fishing line caught in throat,[Entrapped in fishing tackle]
7,Mew Gull. Orphan,[Orphan]
8,American Robin. Cat attack,[Cat interaction]
9,Rock Pigeon. Car hit it,[Vehicle collision]


In [9]:
df.terms.value_counts()

,count
terms,
[Domestic animal interaction],1908
[Orphan],1644
[Cat interaction],1604
[Physical trauma],1338
[Grounded],1324
...,...
"[Orphan, Paint exposure]",1
"[Entrapped in building, Weather event]",1
"[Entrapped in fence, Weather event]",1


In [10]:
df.shape

(36315, 2)

In [11]:
df.text.iloc[100]

'American Robin. HBC- Hit By Car'

## Previous run training data shape

(35845, 3)

In [12]:
df.to_parquet(raw_data_dir/"wildalert_circumstances_api.parquet", index=False)

In [13]:
#df = pd.read_csv(raw_data_dir/"wildalert_circumstances_api.csv") running it if the API fails.

In [22]:
df['terms'] = df['terms'].apply(
    lambda lst: [s.lower().replace(' ', '_') for s in lst] if isinstance(lst, list) else lst
)

In [26]:
all_terms = df['terms'].explode()

all_terms.unique().tolist()

['vehicle_collision',
 'orphan',
 'cat_interaction',
 'entrapped_in_fishing_tackle',
 'non-domestic_animal_interaction',
 'undetermined',
 'trapped_in_leghold_/_trap_/_snare',
 'gunshot',
 'dog_interaction',
 'nest_/_habitat_disturbance_or_destruction',
 'entrapped_in_fence',
 'entrapped_in_building',
 'window_/_wall_collision',
 'entrapped_in_netting_/_string_/_wire',
 'displaced_from_nest',
 'physical_trauma',
 'referral_/_transfer',
 'illness',
 'abduction_with_intent_of_rescue',
 'entrapped_in_litter_/_garbage',
 'hand_held_object_collision',
 'entrapped_in_water',
 'unauthorized_or_untrained_rehabilitation',
 'inappropriate_human_intervention',
 'animal_interaction',
 'bow_and_arrow',
 'fire_/_smoke',
 'electrocution',
 'gas_flare',
 'grease_exposure',
 'petrochemical_exposure',
 'entrapped_in_vehicle',
 'stranded',
 'bicycle_collision',
 'watercraft_collision',
 'maladaptation_/_failure_to_thrive',
 'mating_injury',
 'poisoned',
 'friendly',
 'surrender',
 'wind_turbine_collision

In [27]:
def add_one_hot(df, column):
    _df = df.copy()
    ohe = (pd.get_dummies(_df[column].apply(pd.Series).stack(), dtype="int")
             .groupby(level=0)
             .sum()
          )
    return pd.merge(_df, ohe, left_index=True, right_index=True)

In [28]:
ohe_df = add_one_hot(df, column="terms")

In [29]:
ohe_df.head()

,text,terms,abduction_with_intent_of_rescue,animal_interaction,bicycle_collision,born_in_captivity,botanicals,bow_and_arrow,cat_interaction,collision,...,trapped_in_humane_/_cage_trap,trapped_in_leghold_/_trap_/_snare,tree_trimming,unauthorized_or_untrained_rehabilitation,undetermined,vehicle_collision,watercraft_collision,weather_event,wind_turbine_collision,window_/_wall_collision
0,"Mallard. In road, easy to catch, hit by car. ...",[vehicle_collision],0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,Grey Fox. Orphan,[orphan],0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Malay Spotted Dove. Cat attack,[cat_interaction],0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,"Mew Gull. Found on ground, in road, easy to ca...",[vehicle_collision],0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,Common Merganser. Orphans,[orphan],0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [30]:
ohe_df.columns.tolist()

['text',
 'terms',
 'abduction_with_intent_of_rescue',
 'animal_interaction',
 'bicycle_collision',
 'born_in_captivity',
 'botanicals',
 'bow_and_arrow',
 'cat_interaction',
 'collision',
 'confiscation',
 'cooking_oil_exposure',
 'displaced_from_nest',
 'disturbed_metabolic_rest',
 'dog_interaction',
 'domestic_animal_interaction',
 'dumped',
 'electrocution',
 'entrapment',
 'entrapped_in_building',
 'entrapped_in_chimney',
 'entrapped_in_fence',
 'entrapped_in_fishing_tackle',
 'entrapped_in_litter_/_garbage',
 'entrapped_in_netting_/_string_/_wire',
 'entrapped_in_storm_drain_/_sewer',
 'entrapped_in_vehicle',
 'entrapped_in_water',
 'fire_/_smoke',
 'friendly',
 'garden_/_farm_equipment_collision',
 'gas_flare',
 'grease_exposure',
 'grounded',
 'gunshot',
 'hand_held_object_collision',
 'illness',
 'inappropriate_human_intervention',
 'maladaptation_/_failure_to_thrive',
 'mating_injury',
 'nest_/_habitat_disturbance_or_destruction',
 'non-domestic_animal_interaction',
 'non-wea

In [31]:
ohe_df.to_csv(interim_data_dir/"wildalert_circumstances_api_ohe.csv", index=False)
ohe_df.to_parquet(interim_data_dir/"wildalert_circumstances_api_ohe.parquet", index=False)